# **Notebook 1: Data Preprocessing**

This notebook filters the raw data and prepares it for Exploratory Data Analysis (EDA), Logistic Regression (LR), and Random Forest (RF) / Artificial Neural Network (ANN). It also performs the initial variable reduction from the raw YAFS5 .sav file and the stratified 80/20 train-test split shared by succeeding notebooks.

### **Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import os
import re
import pyreadstat
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

### **Load Raw Dataset**

In [2]:
# Define project directory
PROJECT_ROOT = Path.cwd()

# If the notebook is being run from the notebooks folder, move one level up to the project root
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Define data directories
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

# Define file paths
ORIGINAL_FILE = RAW_DATA_DIR / "YAFS5 Individual.sav"
REDUCED_FILE = PROCESSED_DATA_DIR / "YAFS5_reduced_vars.csv"

# Create processed data directory if it does not exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_ROOT)
print("Original file exists:", ORIGINAL_FILE.exists())

# Load original dataset
raw_df = pd.read_spss(ORIGINAL_FILE)

print("Shape of original dataset:", raw_df.shape)
raw_df.head()

Original file exists: True
Shape of original dataset: (10949, 2248)


,CASEID,HHID,HH1,REG,stratum,psu,URBAN,WGT,WGT1,FWGT,...,j19_flag,k10e_flag,k10f_flag,k10n_flag,abs_abortion,sogie,knowAIDS,N5_count,N5,vaxstat
0,1.0,3.0,3.0,Region 1 (Ilocos Region),2.0,1.0,Rural,0.921926,1679.43,NaN,...,No flag,No flag,No flag,No flag,Does not approve abortion in all circumstances,Heteronormative,No,NaN,NaN,Fully vaccinated
1,2.0,6.0,4.0,Region 1 (Ilocos Region),2.0,1.0,Rural,0.921926,1679.43,1.063418,...,No flag,No flag,No flag,No flag,Does not approve abortion in all circumstances,Heteronormative,No,NaN,NaN,Not vaccinated
2,3.0,7.0,3.0,Region 1 (Ilocos Region),2.0,1.0,Rural,0.921926,1679.43,1.063418,...,No flag,No flag,No flag,No flag,Approves abortion in at least one circumstance,Heteronormative,No,0.0,No,Fully vaccinated
3,4.0,7.0,7.0,Region 1 (Ilocos Region),2.0,1.0,Rural,0.921926,1679.43,NaN,...,No flag,No flag,No flag,No flag,Does not approve abortion in all circumstances,Heteronormative,No,NaN,NaN,Partially vaccinated
4,5.0,8.0,3.0,Region 1 (Ilocos Region),2.0,1.0,Rural,0.921926,1679.43,NaN,...,No flag,No flag,No flag,No flag,Approves abortion in at least one circumstance,Heteronormative,No,NaN,NaN,Partially vaccinated


### **Read SPSS Value-Label Metadata**

In [3]:
_, spss_meta = pyreadstat.read_sav(ORIGINAL_FILE, metadataonly=True)
spss_value_labels = spss_meta.variable_value_labels
print("Columns with SPSS value labels:", len(spss_value_labels))

Columns with SPSS value labels: 978


### **Define Target and Predictor Variables**

In [4]:
# Target variable: CES-D derived depression score
target_var = "cesd"

# Predictor variables
predictor_vars = [
    'G94', 'se_score', 'G95', 'D1', 'B31A', 'G1', 'C5', 'B31F', 'G24', 'I113', 
    'G61', 'sf_score', 'C3', 'I102', 'B4_1', 'E13', 'K26', 'A13', 'B2', 'I133C', 
    'D13_PSCED', 'D10', 'D6', 'I42', 'D37_PSOC', 'L5', 'A9', 'G28', 'J55', 'D46', 
    'C4', 'G87', 'D3', 'E14', 'D32', 'B37C', 'A18', 'B14', 'G16', 'I105', 
    'G32', 'G2', 'G18A', 'I140', 'G17A', 'A8', 'B8A_2', 'B8A_1', 'C1A', 'C1B', 
    'C1C', 'C1D', 'C1E', 'C1F', 'C1G', 'C1H', 'C1I', 'C1J', 'C2A', 'C2B', 
    'C2C', 'C2D', 'C2E', 'C2F', 'C2G', 'C2H', 'D8A', 'D8B', 'D8C', 'D8D', 
    'D8E', 'D8F', 'D8G', 'D8H', 'D8I', 'L12', 'A1', 'A3', 'A3_grp', 'hgc1', 'hgc3', 
    'marstat', 'URBAN', 'urbanex', 'windex', 'windex_score', 'sogie', 'REG', 'A5', 'B1', 'B4_2', 
    'B15', 'B16', 'B17', 'B19F', 'B19G', 'B19C', 'B37A', 'B37E', 'F1', 'F2A', 
    'F2B', 'F5', 'F7', 'F11', 'B37B', 'D9', 'D21', 'D27', 'D29', 'D30', 
    'D51', 'bmi_score', 'bmi_cat', 'G5', 'G7', 'G9', 'G11', 'G14', 'G18D', 'G18K', 
    'G18O', 'G21', 'G27', 'G33', 'G36', 'G44', 'G48', 'G60', 'G68', 'G72', 
    'G56B', 'G101', 'G102', 'G88', 'ever_sex', 'age_first_sex', 'I114', 'J2', 'J4', 'I17', 
    'K6', 'abs_abortion', 'E4', 'E7', 'E10E', 'E10B', 'E1_01', 'E11Q', 'A10', 'A11', 
    'A19', 'vaxstat', 'N13', 'N18B', 'N12D', 'A26F', 'A25', 'A26A', 'A26E', 'A33'
]

print("Number of predictors listed:", len(predictor_vars))

# Check for accidental duplicates in list
duplicates = [v for v in predictor_vars if predictor_vars.count(v) > 1]

if duplicates:
    print("Duplicate entries in predictor_vars:", set(duplicates))
else:
    print("No duplicates found in predictor_vars list.")

Number of predictors listed: 161
No duplicates found in predictor_vars list.


**Verify all variables exist in the dataset**

In [5]:
all_needed = [target_var] + predictor_vars
missing_vars = [col for col in all_needed if col not in raw_df.columns]

if missing_vars:
    raise ValueError(f"Required variables missing from YAFS5: {missing_vars}")

print(f"All {len(all_needed)} required variables exist and found in the dataset.")

All 162 required variables exist and found in the dataset.


### **Subset and Save the Reduced Variable Set**

In [6]:
# Subset to CES-D target variable + 161 selected predictors
reduced_df = raw_df[all_needed].copy()

print("Reduced dataset shape:", reduced_df.shape)
print("Expected shape: (n_rows,", len(all_needed), ")")

os.makedirs(os.path.dirname(REDUCED_FILE), exist_ok=True)
reduced_df.to_csv(REDUCED_FILE, index=False)
print(f"\nReduced dataset saved to: {REDUCED_FILE}")

# Confirm the original .sav file is unmodified
print("Original .sav still present:", os.path.exists(ORIGINAL_FILE))

Reduced dataset shape: (10949, 162)
Expected shape: (n_rows, 162 )

Reduced dataset saved to: /home/abigailching/Thesis/Reduced Dataset/YAFS5_reduced_vars.csv
Original .sav still present: True


**Verify the reduced variable set**

In [7]:
print("Reduced dataset columns:", reduced_df.shape[1])
print("Target variable present:", target_var in reduced_df.columns)
print("All 161 predictors present:", all(v in reduced_df.columns for v in predictor_vars))

# Confirm no unexpected extra columns
extra_cols = set(reduced_df.columns) - set(all_needed)
print("Unexpected extra columns:", extra_cols if extra_cols else "None")

Reduced dataset columns: 162
Target variable present: True
All 161 predictors present: True
Unexpected extra columns: None


### **Load Dataset**

In [8]:
OUT_DIR = "/home/abigailching/Thesis/Outputs"
os.makedirs(OUT_DIR, exist_ok=True)

df = reduced_df.copy()
print("Loaded shape:", df.shape)

df.head()

Loaded shape: (10949, 162)


,cesd,G94,se_score,G95,D1,B31A,G1,C5,B31F,G24,...,A19,vaxstat,N13,N18B,N12D,A26F,A25,A26A,A26E,A33
0,6.0,No,25.0,NaN,"Yes, enrolled",Almost always,1. Very healthy,10 Very happy,Almost always,No,...,No,Fully vaccinated,Yes,Often,0.0,NaN,No,NaN,NaN,Yes
1,9.0,No,17.0,NaN,"Yes, enrolled",Almost always,1. Very healthy,10 Very happy,Almost always,No,...,Yes,Not vaccinated,No,Always,0.0,0.0,Yes,0.0,0.0,Yes
2,10.0,No,18.0,NaN,"Yes, enrolled",Almost always,3. Of average health,8,Frequently,No,...,Yes,Fully vaccinated,Yes,Always,NaN,NaN,No,NaN,NaN,No
3,3.0,No,25.0,NaN,Not enrolled,Almost always,2. Healthier than average,10 Very happy,Almost always,Yes,...,No,Partially vaccinated,Yes,Always,NaN,0.0,Yes,0.0,0.0,Yes
4,8.0,No,17.0,NaN,"Yes, enrolled",Sometimes,1. Very healthy,10 Very happy,Almost always,No,...,No,Partially vaccinated,Yes,Always,0.0,NaN,No,NaN,NaN,No


**Confirm that no numeric-looking columns are still objects**

In [9]:
suspect = [
    c for c in df.columns
    if (df[c].dtype == "object" or isinstance(df[c].dtype, pd.CategoricalDtype))
    and df[c].notna().any()
    and df[c].dropna().astype(str).str.fullmatch(r"-?\d+(\.\d+)?").all()
]

print("Numeric-looking columns still typed as object:", suspect)

Numeric-looking columns still typed as object: []


### **Create Outcome Variable and Define Analytical Samples**

Respondents with **CES-D >11** have **High Depressive Symptoms (HDS)**, while the rest have **Minimal to No Depressive Symptoms (MNDS)**.

The binary outcome variable is created for the full YAFS5 dataset using the CES-D threshold. The complete dataset is retained in a lightweight national context file containing the respondent identifier, region, and outcome for supplementary regional analyses. The primary analytical dataset is then restricted to respondents residing in the National Capital Region (NCR). All subsequent feature selection, model development, model evaluation, and interpretable machine learning procedures use the NCR dataset only.

**Create the binary outcome variable**

In [10]:
raw_df = raw_df.copy()

# Convert CES-D scores to numeric
cesd_numeric = pd.to_numeric(raw_df["cesd"], errors="coerce")
print("Missing or nonnumeric CES-D scores:", cesd_numeric.isna().sum())

raw_df["depressed_flag"] = ((cesd_numeric > 11).astype("Int64").where(cesd_numeric.notna()))

print("\n=== NATIONWIDE OUTCOME DISTRIBUTION ===")
print(raw_df["depressed_flag"].value_counts(dropna=False).sort_index())

Missing or nonnumeric CES-D scores: 0

=== NATIONWIDE OUTCOME DISTRIBUTION ===
depressed_flag
0    9543
1    1406
Name: count, dtype: Int64


**Create the national-context dataset**

In [11]:
national_context_cols = ["REG", "depressed_flag"]

if "CASEID" in raw_df.columns:
    national_context_cols.insert(0, "CASEID")

df_national_context = raw_df[national_context_cols].copy()

print("\n=== NATIONAL CONTEXT DATASET ===")
print("Shape:", df_national_context.shape)
print("Number of regions:", df_national_context["REG"].nunique())

print("\nNationwide outcome distribution:")
print(df_national_context["depressed_flag"].value_counts(dropna=False).sort_index())


=== NATIONAL CONTEXT DATASET ===
Shape: (10949, 3)
Number of regions: 17

Nationwide outcome distribution:
depressed_flag
0    9543
1    1406
Name: count, dtype: Int64


**Save the national-context dataset**

In [12]:
NATIONAL_CONTEXT_OUTPUT = os.path.join(OUT_DIR, "NB1_national_context.csv")
df_national_context.to_csv(NATIONAL_CONTEXT_OUTPUT, index=False)
print(f"National context dataset saved: " f"{NATIONAL_CONTEXT_OUTPUT}")
print(f"Shape: {df_national_context.shape}")
print(f"Columns: {df_national_context.columns.tolist()}")

National context dataset saved: /home/abigailching/Thesis/Outputs/NB1_national_context.csv
Shape: (10949, 3)
Columns: ['CASEID', 'REG', 'depressed_flag']


**Create the NCR primary analytical dataset**

In [13]:
# Create the binary outcome variable in the reduced dataset
cesd_numeric = pd.to_numeric(df["cesd"], errors="coerce")

df["depressed_flag"] = ((cesd_numeric > 11).astype("Int64").where(cesd_numeric.notna()))

# Filter NCR respondents from the reduced dataset only
df_ncr = df.loc[df["REG"] == "National Capital Region (NCR)"].copy()

# Exclude respondents without a valid CES-D outcome
n_before = len(df_ncr)

df_ncr = df_ncr.dropna(subset=["depressed_flag"]).copy()

# Convert outcome to standard integer format
df_ncr["depressed_flag"] = df_ncr["depressed_flag"].astype(int)

n_excluded = n_before - len(df_ncr)

print("\n=== NCR PRIMARY DATASET ===")
print("Original NCR respondents:", n_before)
print("NCR respondents excluded for missing CES-D:", n_excluded)
print("Final NCR dataset shape:", df_ncr.shape)

print(f"\nNCR represents "f"{100 * len(df_ncr) / len(raw_df):.2f}% ""of the nationwide dataset.")

print("\nNCR Class Balance:")
print("1 = High Depressive Symptoms (HDS)")
print("0 = Minimal to No Depressive Symptoms (MNDS)")

print(df_ncr["depressed_flag"].value_counts().sort_index())

print("\nClass distribution (proportions):")
print(df_ncr["depressed_flag"].value_counts(normalize=True).sort_index())

print("\nHigh Depressive Symptoms (HDS):", (df_ncr["depressed_flag"] == 1).sum(), "respondents")
print("Minimal to No Depressive Symptoms (MNDS):", (df_ncr["depressed_flag"] == 0).sum(), "respondents")


=== NCR PRIMARY DATASET ===
Original NCR respondents: 663
NCR respondents excluded for missing CES-D: 0
Final NCR dataset shape: (663, 163)

NCR represents 6.06% of the nationwide dataset.

NCR Class Balance:
1 = High Depressive Symptoms (HDS)
0 = Minimal to No Depressive Symptoms (MNDS)
depressed_flag
0    566
1     97
Name: count, dtype: int64

Class distribution (proportions):
depressed_flag
0    0.853695
1    0.146305
Name: proportion, dtype: float64

High Depressive Symptoms (HDS): 97 respondents
Minimal to No Depressive Symptoms (MNDS): 566 respondents


**Establish the fixed stratified train-test split**

In [14]:
RANDOM_STATE = 42

train_idx, test_idx = train_test_split(
    df_ncr.index, test_size=0.20, random_state=RANDOM_STATE, stratify=df_ncr["depressed_flag"]
)

print("\n=== FIXED TRAIN-TEST SPLIT ===")
print(f"Training observations: {len(train_idx)}")
print(f"Testing observations: {len(test_idx)}")

# Verify expected split dimensions
assert len(train_idx) + len(test_idx) == len(df_ncr)

print("\nTrain-test split established successfully.")


=== FIXED TRAIN-TEST SPLIT ===
Training observations: 530
Testing observations: 133

Train-test split established successfully.


### **Data Summary**

In [15]:
def data_summary(frame, label, target='depressed_flag'):
    print("=" * 66); print(f"DATA SUMMARY: {label}"); print("=" * 66)
    print(f"Shape: {frame.shape[0]} rows x {frame.shape[1]} columns")
    
    if target in frame.columns:
        vc = frame[target].value_counts(); prop = frame[target].value_counts(normalize=True)
        print(f"Target (1=HDS, 0=MNDS): HDS={int(vc.get(1,0))} ({prop.get(1,0)*100:.1f}%) | "
        f"MNDS={int(vc.get(0,0))} ({prop.get(0,0)*100:.1f}%)")
    
    print("Dtypes:", {str(k): int(v) for k, v in frame.dtypes.astype(str).value_counts().items()}, "\n")
    miss = frame.isna().sum(); miss = miss[miss > 0].sort_values(ascending=False)
    print(f"Total missing cells: {int(frame.isna().sum().sum())} across {len(miss)} columns")
    
    for c, n in miss.items():
        print(f"{c}: {int(n)} ({100*n/len(frame):.1f}%)")

data_summary(df_ncr, "Before Preprocessing")

DATA SUMMARY: Before Preprocessing
Shape: 663 rows x 163 columns
Target (1=HDS, 0=MNDS): HDS=97 (14.6%) | MNDS=566 (85.4%)
Dtypes: {'category': 139, 'float64': 23, 'int64': 1} 

Total missing cells: 18034 across 66 columns
G72: 649 (97.9%)
J55: 637 (96.1%)
K26: 592 (89.3%)
J4: 592 (89.3%)
N12D: 588 (88.7%)
G32: 587 (88.5%)
G28: 587 (88.5%)
D13_PSCED: 579 (87.3%)
J2: 552 (83.3%)
L5: 548 (82.7%)
E14: 539 (81.3%)
A26E: 535 (80.7%)
A26A: 535 (80.7%)
A26F: 535 (80.7%)
G95: 497 (75.0%)
D37_PSOC: 475 (71.6%)
G27: 474 (71.5%)
I42: 453 (68.3%)
G36: 429 (64.7%)
I102: 413 (62.3%)
age_first_sex: 413 (62.3%)
G56B: 410 (61.8%)
B19G: 387 (58.4%)
B19F: 387 (58.4%)
B19C: 387 (58.4%)
G88: 276 (41.6%)
D30: 275 (41.5%)
D6: 261 (39.4%)
D3: 261 (39.4%)
D8G: 255 (38.5%)
D8D: 255 (38.5%)
D8I: 255 (38.5%)
D8F: 255 (38.5%)
D8E: 255 (38.5%)
D8A: 255 (38.5%)
D8H: 255 (38.5%)
D8B: 255 (38.5%)
D8C: 255 (38.5%)
D32: 253 (38.2%)
I105: 243 (36.7%)
G11: 243 (36.7%)
A10: 185 (27.9%)
G60: 154 (23.2%)
G48: 154 (23.2%)
G61

### **Export as CSV the NCR-Filtered Data**

In [16]:
checkpoint1 = os.path.join(OUT_DIR, "NB1_filtered_raw.csv")
df_ncr.to_csv(checkpoint1, index=False)
print(f"Checkpoint saved: {checkpoint1}")

Checkpoint saved: /home/abigailching/Thesis/Outputs/NB1_filtered_raw.csv


### **Drop Geography-Constant Columns**

This step was done because columns related to geography (e.g., urbanization) are not relevant anymore because our data has been filtered to only include those from NCR.

In [17]:
geography_constant_cols = ['REG', 'URBAN', 'urbanex']
df_ncr = df_ncr.drop(columns=geography_constant_cols)
print("Dropped geography-constant columns:", geography_constant_cols)
print("Shape after drop:", df_ncr.shape)

Dropped geography-constant columns: ['REG', 'URBAN', 'urbanex']
Shape after drop: (663, 160)


### **Handle Nonresponse Values**

Certain YAFS5 responses are stored as text labels rather than actual missing values. Confirmed nonresponse labels are converted to `NaN` to ensure they are appropriately handled during subsequent preprocessing. Parenthesized responses that may represent valid or structurally applicable responses are retained unless their treatment has been verified.

In [18]:
# Explicitly identified nonresponse labels
nonresponse_labels = {"don't know", "refused to answer", "no information"}

# Identify categorical and object columns
obj_cols = df_ncr.select_dtypes(include=["object", "category"]).columns.tolist()

before = df_ncr[obj_cols].isna().sum().sum()
recoded_labels = {}

for col in obj_cols:
    s = df_ncr[col].astype(object)

    # Identify confirmed nonresponse labels
    hit = s.apply(lambda v: (isinstance(v, str) and v.strip().strip("()").strip().casefold() in nonresponse_labels)
                 ).fillna(False).astype(bool)

    if hit.any():
        recoded_labels[col] = s[hit].value_counts().to_dict()
        # Recode confirmed nonresponse values as missing
        df_ncr[col] = s.mask(hit, np.nan)

after = df_ncr[obj_cols].isna().sum().sum()

print("=== NONRESPONSE VALUE RECODING ===")
print("Columns checked:", len(obj_cols))
print("Missing cells before:", before)
print("Missing cells after:", after)
print("Newly recoded missing values:", after - before)

print("\nColumns with nonresponse labels recoded:")

for col, labs in recoded_labels.items():
    print(f"  {col}: {labs}")

=== NONRESPONSE VALUE RECODING ===
Columns checked: 136
Missing cells before: 12665
Missing cells after: 12715
Newly recoded missing values: 50

Columns with nonresponse labels recoded:
  G94: {'(Refused to answer)': 3}
  G95: {'(Refused to answer)': 1}
  B4_1: {"(Don't know)": 8}
  B15: {'(No information)': 29}
  D51: {'(No information)': 4}
  G5: {"(Don't know)": 2, '(Refused to answer)': 1}
  G33: {'(Refused to answer)': 1}
  G68: {'(Refused to answer)': 1}


**Remove redundant predictors**

In [19]:
redundant_cols_to_drop = [
    "A3_grp", "hgc1",
    "windex", "bmi_cat"
]

before_shape = df_ncr.shape

df_ncr = df_ncr.drop(
    columns=[c for c in redundant_cols_to_drop if c in df_ncr.columns])

print("Dropped redundant columns:", redundant_cols_to_drop)
print("Shape before:", before_shape, "-> after:", df_ncr.shape)

Dropped redundant columns: ['A3_grp', 'hgc1', 'windex', 'bmi_cat']
Shape before: (663, 160) -> after: (663, 156)


### **Identify Mixed Numeric/Text Columns**

In [20]:
candidate_cols = df_ncr.select_dtypes(include="object").columns.tolist()

mixed_format_cols = []

for col in candidate_cols:
    sv = (df_ncr.loc[train_idx, col].dropna().astype(str).str.strip())
    has_pure_numeric = sv.str.match(r"^\d+(\.\d+)?$").any()
    has_number_with_label = sv.str.match(r"^\d+(\.\d+)?\s+\S").any()
    has_pure_text = sv.str.match(r"^[A-Za-z]").any()

    if has_number_with_label or (has_pure_numeric and has_pure_text):
        mixed_format_cols.append(col)

print("Columns with mixed numeric/text formats:", mixed_format_cols)

Columns with mixed numeric/text formats: []


### **Handling Mixed Numeric/Text Columns**

Certain YAFS5 variables contain a combination of numerical responses and text labels. These variables are reviewed and transformed where appropriate to ensure compatibility with subsequent analytical procedures.

Special text responses are first recoded based on their meaning. Mixed-format columns are then evaluated using the training partition to determine whether their values can be safely converted into numeric representations without merging distinct response categories. Variables that cannot be safely converted are retained as categorical variables.

**Special text value recoding**

In [21]:
# G14: Less than a minute -> 0.5 minutes
df_ncr["G14"] = df_ncr["G14"].replace({"Less than a minute": 0.5})

# E7: Less than 15 minutes -> 0.125 hours
df_ncr["E7"] = df_ncr["E7"].replace({"Less than 15 minutes": 0.125})

# G28: Occasional smoker -> missing
df_ncr["G28"] = df_ncr["G28"].replace({"Occasional smoker": np.nan})

df_ncr["G28"] = pd.to_numeric(df_ncr["G28"], errors="coerce")

# B8A_1 and B8A_2: Invalid response code
for col in ["B8A_1", "B8A_2"]:
    df_ncr[col] = df_ncr[col].replace({99997.0: np.nan})

# E10B and E10E: None (0) -> 0
for col in ["E10B", "E10E"]:
    df_ncr[col] = df_ncr[col].replace({"None (0)": 0})

print("Special text-value recoding completed.")

/tmp/ipykernel_17459/999287513.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_ncr["G14"] = df_ncr["G14"].replace({"Less than a minute": 0.5})
/tmp/ipykernel_17459/999287513.py:2: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df_ncr["G14"] = df_ncr["G14"].replace({"Less than a minute": 0.5})
/tmp/ipykernel_17459/999287513.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior

Special text-value recoding completed.


/tmp/ipykernel_17459/999287513.py:18: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df_ncr[col] = df_ncr[col].replace({"None (0)": 0})


**Save pre-learning checkpoint**

In [22]:
NB1_PRELEARN = os.path.join(OUT_DIR, "NB1_prelearn_cleaned_NCR.pkl")

pd.to_pickle({
    "data": df_ncr.copy(), "spss_value_labels": spss_value_labels,
    "train_idx": list(train_idx), "test_idx": list(test_idx), }, NB1_PRELEARN)

print("Saved fold-refitting input:", NB1_PRELEARN)

Saved fold-refitting input: /home/abigailching/Thesis/Outputs/NB1_prelearn_cleaned_NCR.pkl


**Function to extract numbers**

In [23]:
def extract_number(val):
    if pd.isna(val):
        return np.nan

    if isinstance(val, (int, float)):
        return float(val)

    match = re.search(r'\d+(\.\d+)?', str(val))
    return float(match.group()) if match else np.nan

cols_to_clean = [
    "B8A_1", "B8A_2", "C3", "C4", "C5", 
    "G14", "G28", "E7", "E10B", "E10E",
]

def safe_numeric_convert(series, min_coverage=1.0):
    """Convert a mixed numeric/text column to float only when it is lossless:
    (a) all non-missing values contain a number, and
    (b) every distinct label maps to its own distinct number (nothing gets merged)."""
    s = series.astype(object)
    nn = s.dropna()
    parsed = nn.apply(extract_number)
    coverage = float(parsed.notna().mean()) if len(nn) else 0.0
    kept = nn[parsed.notna()]
    n_labels = kept.astype(str).nunique()
    n_numbers = parsed.dropna().nunique()
    lossless = (coverage >= min_coverage and n_numbers == n_labels)
    dropped = sorted(set(nn[parsed.isna()].astype(str)))
    if lossless:
        return s.apply(extract_number).astype(float), coverage, True, dropped
    return series, coverage, False, dropped

candidates = [c for c in dict.fromkeys(cols_to_clean + mixed_format_cols) if c in df_ncr.columns]
converted, kept_categorical = [], []
for col in candidates:
    if pd.api.types.is_numeric_dtype(df_ncr[col]):
        continue
    new_col, cov, ok, dropped = safe_numeric_convert(df_ncr.loc[train_idx, col])
    if ok:
        df_ncr[col] = df_ncr[col].apply(extract_number).astype(float)
        converted.append(col)
        extra = f" | labels with no number (now NaN): {dropped}" if dropped else ""
        print(f"  CONVERTED to numeric: {col} (coverage {cov:.1%}){extra}")
    else:
        kept_categorical.append(col)
        print(f"  kept categorical:     {col} (coverage {cov:.1%}, labels would merge or are not numeric)")

print("\nConverted to numeric:", converted)
print("Kept categorical (ordered by SPSS code when encoded):", kept_categorical)

  kept categorical:     B8A_1 (coverage 40.3%, labels would merge or are not numeric)
  kept categorical:     B8A_2 (coverage 41.5%, labels would merge or are not numeric)
  CONVERTED to numeric: C3 (coverage 100.0%)
  CONVERTED to numeric: C4 (coverage 100.0%)
  CONVERTED to numeric: C5 (coverage 100.0%)
  CONVERTED to numeric: G14 (coverage 100.0%)
  CONVERTED to numeric: E7 (coverage 100.0%)
  CONVERTED to numeric: E10B (coverage 100.0%)
  CONVERTED to numeric: E10E (coverage 100.0%)

Converted to numeric: ['C3', 'C4', 'C5', 'G14', 'E7', 'E10B', 'E10E']
Kept categorical (ordered by SPSS code when encoded): ['B8A_1', 'B8A_2']


### **Identify High-Cardinality, Non-Numeric Columns**

Columns with more than 8 unique values are identified.

In [24]:
predictor_cols = [c for c in df_ncr.columns if c not in ['cesd', 'depressed_flag']]
high_card_non_numeric = []
for col in predictor_cols:
    if not pd.api.types.is_numeric_dtype(df_ncr[col]) and df_ncr.loc[train_idx, col].nunique(dropna=True) > 8:
        high_card_non_numeric.append(col)
print(f"Found {len(high_card_non_numeric)} high-cardinality non-numeric columns:")
print(high_card_non_numeric)

Found 9 high-cardinality non-numeric columns:
['D37_PSOC', 'L5', 'D3', 'B37C', 'B8A_2', 'B8A_1', 'B37A', 'B37E', 'B37B']


### **Recode Rare Categories to "Others"**

In [25]:
RARE_THRESHOLD = 0.01
def recode_rare_categories(series, threshold=RARE_THRESHOLD, fit_series=None):
    fit_series = series if fit_series is None else fit_series
    freq = fit_series.value_counts(normalize=True, dropna=True)
    rare_levels = freq[freq < threshold].index.tolist()
    return series.apply(lambda x: "Others" if x in rare_levels else x)

for col in high_card_non_numeric:
    b = df_ncr[col].nunique(dropna=True)
    df_ncr[col] = recode_rare_categories(df_ncr[col], fit_series=df_ncr.loc[train_idx, col])
    print(f"{col}: {b} -> {df_ncr[col].nunique(dropna=True)} unique values")

D37_PSOC: 27 -> 23 unique values
L5: 16 -> 16 unique values
D3: 15 -> 11 unique values
B37C: 11 -> 8 unique values
B8A_2: 23 -> 14 unique values
B8A_1: 24 -> 16 unique values
B37A: 14 -> 8 unique values
B37E: 11 -> 8 unique values
B37B: 11 -> 8 unique values


### **Fix Confirmed Skip-Pattern Missingness**

Skip-pattern variables are variables that are only asked if a respondent gives a certain answer to a previous question.

In [26]:
pd.crosstab(df_ncr["ever_sex"], df_ncr["age_first_sex"].isna())

age_first_sex,False,True
ever_sex,,
No,0,413
Yes,250,0


Skip-pattern missingness is confirmed because:
- Every respondent who answered "No" to ever_sex has a missing age_first_sex.
- Every respondent who answered "Yes" has a recorded age_first_sex.

**Apply fix for skip-pattern variable**

In [27]:
never_had_sex = df_ncr['ever_sex'] == 'No'
print(f"Respondents gated out of age_first_sex: {never_had_sex.sum()}")

# Structural recode: never had sex -> age_first_sex = 0.
df_ncr.loc[never_had_sex, 'age_first_sex'] = 0

# True nonresponse
n_true_nr = (df_ncr['age_first_sex'].isna() & ~never_had_sex).sum()
print(f"Remaining age_first_sex: {n_true_nr}")

print("\nGated fixes applied.")

Respondents gated out of age_first_sex: 413
Remaining age_first_sex: 0

Gated fixes applied.


### **Save EDA-Friendly Version**

In [28]:
NB1_EDA_OUTPUT = os.path.join(OUT_DIR, "NB1_data_EDA.csv")
df_eda = df_ncr.drop(columns=['cesd'])
df_eda.to_csv(NB1_EDA_OUTPUT, index=False)
print(f"EDA checkpoint saved: {NB1_EDA_OUTPUT}")
print(f"Shape: {df_eda.shape}")
print(f"cesd present: {'cesd' in df_eda.columns}")
print(f"depressed_flag present: {'depressed_flag' in df_eda.columns}")
print(f"Remaining missing cells: "
      f"{df_eda[[c for c in df_eda.columns if c != 'depressed_flag']].isnull().sum().sum()}")

EDA checkpoint saved: /home/abigailching/Thesis/Outputs/NB1_data_EDA.csv
Shape: (663, 155)
cesd present: False
depressed_flag present: True
Remaining missing cells: 17723


### **Prepare Random Forest (RF) / Artificial Neural Network (ANN) Dataset**

This code label-encodes categorical columns in their SPSS code order (so ordinal variables keep their true order instead of being sorted alphabetically) and introduces a sentinel value (specifically -1) for missing values.

Category-code levels are learned from training respondents only and then applied to the held-out test respondents; unseen test labels become the existing `-1` sentinel.

**Initialize RF/ANN dataset**

In [29]:
SENTINEL = -1.0

df_rfann = df_ncr.drop(columns=["cesd"]).copy()

print("RF/ANN dataset initialized.")
print("Shape:", df_rfann.shape)
print("Target variable retained:", "depressed_flag" in df_rfann.columns)

RF/ANN dataset initialized.
Shape: (663, 155)
Target variable retained: True


**Define categorical level ordering**

In [30]:
def ordered_levels(col, levels):
    lab2code = {}

    for code, lab in spss_value_labels.get(col, {}).items():
        try:
            lab2code[str(lab)] = float(code)
        except (TypeError, ValueError):
            pass

    def key(lvl):
        if lvl in lab2code:
            return (0, lab2code[lvl], lvl)

        try:
            return (0, float(lvl), lvl)
        except ValueError:
            return (2, 0.0, lvl) if lvl == "Others" else (1, 0.0, lvl)

    return sorted(levels, key=key)

print("Categorical level ordering function defined.")

Categorical level ordering function defined.


**Identify categorical variables**

In [31]:
cat_cols = [
    c for c in df_rfann.columns
    if c != "depressed_flag" and not pd.api.types.is_numeric_dtype(df_rfann[c])
]

print("Number of categorical variables:", len(cat_cols))
print("\nCategorical variables:")
print(cat_cols)

Number of categorical variables: 124

Categorical variables:
['G94', 'G95', 'D1', 'B31A', 'G1', 'B31F', 'G24', 'I113', 'G61', 'I102', 'B4_1', 'E13', 'K26', 'A13', 'B2', 'I133C', 'D13_PSCED', 'D10', 'D6', 'D37_PSOC', 'L5', 'A9', 'D46', 'G87', 'D3', 'E14', 'D32', 'B37C', 'A18', 'B14', 'I105', 'G32', 'G2', 'G18A', 'I140', 'A8', 'B8A_2', 'B8A_1', 'C1A', 'C1B', 'C1C', 'C1D', 'C1E', 'C1F', 'C1G', 'C1H', 'C1I', 'C1J', 'C2A', 'C2B', 'C2C', 'C2D', 'C2E', 'C2F', 'C2G', 'C2H', 'D8A', 'D8B', 'D8C', 'D8D', 'D8E', 'D8F', 'D8G', 'D8H', 'D8I', 'L12', 'A1', 'hgc3', 'marstat', 'sogie', 'A5', 'B4_2', 'B15', 'B16', 'B17', 'B37A', 'B37E', 'F1', 'F5', 'F7', 'F11', 'B37B', 'D9', 'D21', 'D27', 'D29', 'D30', 'D51', 'G5', 'G7', 'G9', 'G11', 'G18D', 'G18K', 'G18O', 'G21', 'G27', 'G33', 'G36', 'G44', 'G48', 'G60', 'G68', 'G72', 'G101', 'G102', 'G88', 'ever_sex', 'I114', 'J2', 'J4', 'I17', 'K6', 'abs_abortion', 'E4', 'E1_01', 'E11Q', 'A10', 'A19', 'vaxstat', 'N13', 'N18B', 'A25', 'A33']


**Label encode categorical variables**

In [32]:
level_orders = {}

for col in cat_cols:
    # Convert categorical dtype to object
    s = df_rfann[col].astype(object)

    # Obtain observed training levels
    levels = ordered_levels(col, s.loc[train_idx].dropna().astype(str).unique().tolist())

    # Assign numerical codes
    mapping = {lvl: float(i) for i, lvl in enumerate(levels)}
    level_orders[col] = levels

    # Apply mapping to both training and testing partitions
    df_rfann[col] = s.apply(
        lambda x: mapping.get(str(x), SENTINEL)
        if pd.notna(x)
        else SENTINEL
    )

print(f"Label-encoded {len(cat_cols)} categorical variables.")

# Preview selected mappings
for col in cat_cols[:5]:
    print(f"\n{col}: {level_orders[col][:6]}")

# Verify that RF/ANN categorical encoding is numeric
assert all(
    pd.api.types.is_numeric_dtype(df_rfann[col])
    for col in cat_cols
)

print("All categorical variables successfully encoded.")

Label-encoded 124 categorical variables.

G94: ['No', 'Yes, once', 'Yes, more than once']

G95: ['No', 'Yes, once', 'Yes, more than once']

D1: ['Yes, enrolled', 'Yes, enrolled but dropped out', 'Not enrolled', 'Never been enrolled']

B31A: ['Almost never', 'Once in a while', 'Sometimes', 'Frequently', 'Almost always']

G1: ['1. Very healthy', '2. Healthier than average', '3. Of average health', '4. Somewhat unhealthy']
All categorical variables successfully encoded.


**Handle missing numerical values**

In [33]:
num_cols = [
    c for c in df_rfann.columns
    if c != "depressed_flag" and pd.api.types.is_numeric_dtype(df_rfann[c])]

for col in num_cols:
    df_rfann[col] = df_rfann[col].fillna(SENTINEL)

print("Numerical variables processed:", len(num_cols))
print("Missing numerical values replaced with:", SENTINEL)

Numerical variables processed: 154
Missing numerical values replaced with: -1.0


**Validate dataset**

In [34]:
# Check for remaining missing values
remaining_missing = (df_rfann.drop(columns=["depressed_flag"]).isna().sum().sum())

# Check that all columns are numerical
all_numeric = all(pd.api.types.is_numeric_dtype(df_rfann[c]) for c in df_rfann.columns)

assert remaining_missing == 0, "NaN remains in RF/ANN predictors."
assert all_numeric, "Non-numeric column remains."

print("=== RF/ANN DATASET VALIDATION ===")
print("Shape:", df_rfann.shape)
print("Remaining missing predictor values:", remaining_missing)
print("All columns numeric:", all_numeric)

=== RF/ANN DATASET VALIDATION ===
Shape: (663, 155)
Remaining missing predictor values: 0
All columns numeric: True


**Export categorical mappings**

In [35]:
level_order_df = pd.DataFrame([
    {"feature": c, "code": i, "level": lvl}
    for c, lvls in level_orders.items()
    for i, lvl in enumerate(lvls)
])

LEVEL_ORDER_OUTPUT = os.path.join(OUT_DIR, "NB1_RF_ANN_level_order.csv")
level_order_df.to_csv(LEVEL_ORDER_OUTPUT, index=False)

print("Categorical level mapping saved:", LEVEL_ORDER_OUTPUT)

Categorical level mapping saved: /home/abigailching/Thesis/Outputs/NB1_RF_ANN_level_order.csv


**Export final dataset**

In [36]:
NB1_RFANN_OUTPUT = os.path.join(OUT_DIR, "NB1_data_prepared_RF_ANN.csv")
df_rfann.to_csv(NB1_RFANN_OUTPUT, index=False)

print("RF/ANN dataset saved:", NB1_RFANN_OUTPUT)
print("Final shape:", df_rfann.shape)
print("Remaining missing values:", df_rfann.isna().sum().sum())
print("All columns numeric:", all_numeric)

RF/ANN dataset saved: /home/abigailching/Thesis/Outputs/NB1_data_prepared_RF_ANN.csv
Final shape: (663, 155)
Remaining missing values: 0
All columns numeric: True


### **One-Hot Encoding (OHE) for All Categorical Columns for Logistic Regression (LR)**
OHE vocabulary is learned from training respondents only (`dummy_na=False`); unseen test categories generate no new dummy columns.

In [37]:
cat_cols = [c for c in df_ncr.select_dtypes(include=['object', 'category']).columns
            if c not in ['cesd']]

# Fit OHE columns using training rows only
train_ohe = pd.get_dummies(df_ncr.loc[train_idx], columns=cat_cols,
                           dummy_na=False, dtype=int)
df_ncr_encoded = pd.get_dummies(df_ncr, columns=cat_cols, 
                                dummy_na=False, dtype=int).reindex(columns=train_ohe.columns, fill_value=0)

assert df_ncr_encoded.loc[train_idx].equals(train_ohe)

print(f"Shape before OHE: {df_ncr.shape}")
print(f"Shape after OHE: {df_ncr_encoded.shape}")
print(f"New columns added: {df_ncr_encoded.shape[1] - df_ncr.shape[1]}")

Shape before OHE: (663, 156)
Shape after OHE: (663, 584)
New columns added: 428


### **Residual Numeric Missingness Report Before Sentinel-Encoding**

In [38]:
predictor_cols_encoded = [c for c in df_ncr_encoded.columns if c not in ['cesd', 'depressed_flag']]
df_ncr_handled = df_ncr_encoded.copy()

numeric_remaining = [c for c in predictor_cols_encoded
                     if pd.api.types.is_numeric_dtype(df_ncr_handled[c])
                     and df_ncr_handled[c].isna().sum() > 0]

print(f"Numeric columns with residual missingness (-> sentinel(-1) later): {len(numeric_remaining)}")

for col in numeric_remaining:
    n = df_ncr_handled[col].isna().sum()
    print(f"  {col}: {n} missing ({100*n/len(df_ncr_handled):.2f}%)")
    
print(f"\nTotal numeric cells to sentinel-encode: {df_ncr_handled[predictor_cols_encoded].isnull().sum().sum()}")

# Total predictor columns after OHE
predictor_cols = [c for c in df_ncr_handled.columns if c not in ['cesd', 'depressed_flag']]
print(f"Total predictor columns after one-hot encoding: {len(predictor_cols)}")

Numeric columns with residual missingness (-> sentinel(-1) later): 17
  I42: 453 missing (68.33%)
  G28: 589 missing (88.84%)
  J55: 637 missing (96.08%)
  B19F: 387 missing (58.37%)
  B19G: 387 missing (58.37%)
  B19C: 387 missing (58.37%)
  F2A: 49 missing (7.39%)
  F2B: 49 missing (7.39%)
  G56B: 410 missing (61.84%)
  E7: 8 missing (1.21%)
  E10E: 8 missing (1.21%)
  E10B: 8 missing (1.21%)
  A11: 4 missing (0.60%)
  N12D: 588 missing (88.69%)
  A26F: 535 missing (80.69%)
  A26A: 535 missing (80.69%)
  A26E: 535 missing (80.69%)

Total numeric cells to sentinel-encode: 5569
Total predictor columns after one-hot encoding: 582


### **Sentinel Value for Residual Numeric Missing Values**

In [39]:
SENTINEL = -1.0
num_with_nan = [c for c in df_ncr_handled.columns
                if c != 'depressed_flag' and df_ncr_handled[c].isna().any()]

for col in num_with_nan:
    n = int(df_ncr_handled[col].isna().sum())
    if isinstance(df_ncr_handled[col].dtype, pd.CategoricalDtype):
        df_ncr_handled[col] = df_ncr_handled[col].astype(object)   # <-- break out of category dtype
    df_ncr_handled[col] = df_ncr_handled[col].fillna(SENTINEL)
    print(f"  {col}: {n} missing -> sentinel(-1)")

assert df_ncr_handled.drop(columns=['depressed_flag']).isna().sum().sum() == 0, "NaN remains in LR file"
print("Remaining Missing Values:", df_ncr_handled.isnull().sum().sum())

  I42: 453 missing -> sentinel(-1)
  G28: 589 missing -> sentinel(-1)
  J55: 637 missing -> sentinel(-1)
  B19F: 387 missing -> sentinel(-1)
  B19G: 387 missing -> sentinel(-1)
  B19C: 387 missing -> sentinel(-1)
  F2A: 49 missing -> sentinel(-1)
  F2B: 49 missing -> sentinel(-1)
  G56B: 410 missing -> sentinel(-1)
  E7: 8 missing -> sentinel(-1)
  E10E: 8 missing -> sentinel(-1)
  E10B: 8 missing -> sentinel(-1)
  A11: 4 missing -> sentinel(-1)
  N12D: 588 missing -> sentinel(-1)
  A26F: 535 missing -> sentinel(-1)
  A26A: 535 missing -> sentinel(-1)
  A26E: 535 missing -> sentinel(-1)
Remaining Missing Values: 0


### **Drop CESD Column**

In [40]:
df_ncr_handled = df_ncr_handled.drop(columns=['cesd'])
predictor_cols = [c for c in df_ncr_handled.columns if c not in ['cesd', 'depressed_flag']]
print("cesd dropped.")
print("Final shape:", df_ncr_handled.shape)

cesd dropped.
Final shape: (663, 583)


### **Data Summary (After Preprocessing)**

In [41]:
data_summary(df_ncr_handled, "LR file (one-hot) - NB1_data_prepared_NCR.csv")
data_summary(df_rfann, "RF/ANN file (label-encoded + sentinel) - NB1_data_prepared_RF_ANN.csv")

DATA SUMMARY: LR file (one-hot) - NB1_data_prepared_NCR.csv
Shape: 663 rows x 583 columns
Target (1=HDS, 0=MNDS): HDS=97 (14.6%) | MNDS=566 (85.4%)
Dtypes: {'int64': 553, 'float64': 30} 

Total missing cells: 0 across 0 columns
DATA SUMMARY: RF/ANN file (label-encoded + sentinel) - NB1_data_prepared_RF_ANN.csv
Shape: 663 rows x 155 columns
Target (1=HDS, 0=MNDS): HDS=97 (14.6%) | MNDS=566 (85.4%)
Dtypes: {'float64': 154, 'int64': 1} 

Total missing cells: 0 across 0 columns


### **Save File for Logistic Regression**

In [42]:
NB1_OUTPUT = os.path.join(OUT_DIR, "NB1_data_prepared_NCR.csv")
df_ncr_handled.to_csv(NB1_OUTPUT, index=False)

print(f"Model-ready output saved: {NB1_OUTPUT}")
print(f"Shape: {df_ncr_handled.shape}")

Model-ready output saved: /home/abigailching/Thesis/Outputs/NB1_data_prepared_NCR.csv
Shape: (663, 583)


### **Stratified Train-Test Split (80/20 Split)**

In [43]:
RANDOM_STATE = 42

assert df_rfann.index.equals(df_ncr_handled.index), \
    "LR and RF/ANN frames must share the same row index before splitting"

y_split = df_ncr_handled['depressed_flag'].copy()

assert set(train_idx).isdisjoint(set(test_idx))
assert set(train_idx).union(set(test_idx)) == set(df_ncr_handled.index)

print(f"Train: {len(train_idx)} | Test: {len(test_idx)}")
print("Train balance:\n", y_split.loc[train_idx].value_counts(normalize=True).round(4))
print("Test balance:\n",  y_split.loc[test_idx].value_counts(normalize=True).round(4))

Train: 530 | Test: 133
Train balance:
 depressed_flag
0    0.8528
1    0.1472
Name: proportion, dtype: float64
Test balance:
 depressed_flag
0    0.8571
1    0.1429
Name: proportion, dtype: float64


### **Save Train/Test Splits (LR and RF/ANN)**

In [44]:
LR_TRAIN_OUT = os.path.join(OUT_DIR, "NB1_train_LR.csv")
LR_TEST_OUT = os.path.join(OUT_DIR, "NB1_test_LR.csv")
RFANN_TRAIN_OUT = os.path.join(OUT_DIR, "NB1_train_RF_ANN.csv")
RFANN_TEST_OUT = os.path.join(OUT_DIR, "NB1_test_RF_ANN.csv")

df_ncr_handled.loc[train_idx].to_csv(LR_TRAIN_OUT, index=True, index_label="orig_index")
df_ncr_handled.loc[test_idx].to_csv(LR_TEST_OUT, index=True, index_label="orig_index")
df_rfann.loc[train_idx].to_csv(RFANN_TRAIN_OUT, index=True, index_label="orig_index")
df_rfann.loc[test_idx].to_csv(RFANN_TEST_OUT, index=True, index_label="orig_index")

print("Saved train/test splits:")
print(" ", LR_TRAIN_OUT, df_ncr_handled.loc[train_idx].shape)
print(" ", LR_TEST_OUT, df_ncr_handled.loc[test_idx].shape)
print(" ", RFANN_TRAIN_OUT, df_rfann.loc[train_idx].shape)
print(" ", RFANN_TEST_OUT, df_rfann.loc[test_idx].shape)

Saved train/test splits:
  /home/abigailching/Thesis/Outputs/NB1_train_LR.csv (530, 583)
  /home/abigailching/Thesis/Outputs/NB1_test_LR.csv (133, 583)
  /home/abigailching/Thesis/Outputs/NB1_train_RF_ANN.csv (530, 155)
  /home/abigailching/Thesis/Outputs/NB1_test_RF_ANN.csv (133, 155)
